# CP2K band unfolding

This viewer loads precomputed unfolding weights retrieved by the CP2K SCF workchain.


In [ ]:
%load_ext aiida
%aiida

import urllib.parse as urlparse

import ipywidgets as ipw
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from aiida import orm


In [ ]:
def _query_pk(default=None):
    try:
        query = urlparse.parse_qs(urlparse.urlparse(jupyter_notebook_url).query)
        if "pk" in query:
            return int(query["pk"][0])
    except Exception:
        pass
    return default

pk = _query_pk()
if pk is None:
    raise ValueError("Open this viewer from a CP2K SCF unfolding result, or pass ?pk=<workchain_pk>.")

workcalc = orm.load_node(pk)
if "unfolding_retrieved" not in workcalc.outputs:
    raise ValueError("This SCF workchain has no unfolding_retrieved output. Submit with Compute band unfolding enabled.")

with workcalc.outputs.unfolding_retrieved.open("unfolding_bands.npz", "rb") as handle:
    loaded = np.load(handle)
    data = {key: loaded[key] for key in loaded.files}

spin_indices = sorted(
    int(key.rsplit("_", 1)[-1])
    for key in data
    if key.startswith("weights_spin_")
)

print(f"SCF workchain PK: {workcalc.pk}")
print("label:", workcalc.label)
print("primitive vectors [Angstrom]:")
print(data["primitive_vectors"])
print("supercell matrix M:")
print(data["supercell_matrix"])
print("det(M):", int(round(abs(np.linalg.det(data["supercell_matrix"])))))
print("folded k-points:", len(data["k_frac_folded"]))
print("k-points on path:", len(data["path_k_indices"]))
print("path:", "-".join(data["path_labels"].astype(str)))


In [ ]:
def _canonical_frac(values, tol=1.0e-8):
    folded = np.mod(np.asarray(values, dtype=float), 1.0)
    folded[np.isclose(folded, 1.0, atol=tol) | np.isclose(folded, 0.0, atol=tol)] = 0.0
    return folded


def _lattice_matrix(vectors):
    vec = np.asarray(vectors, dtype=float)
    dim = int(data["dim"])
    return vec[:dim, :dim].T


def _reciprocal_vectors(primitive_vectors):
    return 2.0 * np.pi * np.linalg.inv(_lattice_matrix(primitive_vectors)).T


def _kfrac_to_cart(k_frac, primitive_vectors):
    k_frac = np.asarray(k_frac, dtype=float)
    bmat = _reciprocal_vectors(primitive_vectors)
    k_dim = k_frac @ bmat.T
    cart = np.zeros((len(k_dim), 3))
    cart[:, : bmat.shape[0]] = k_dim
    return cart


def _display_high_symmetry_points():
    dim = int(data["dim"])
    lattice_type = str(np.asarray(data.get("lattice_type", "")).item()).lower()
    if dim == 1:
        return {"G": np.array([0.0]), "X": np.array([0.5])}
    if lattice_type in {"hex", "hexagonal", "graphene"}:
        return {
            "G": np.array([0.0, 0.0]),
            "K": np.array([1.0 / 3.0, 1.0 / 3.0]),
            "M": np.array([0.5, 0.0]),
        }
    points = {}
    for label in data["path_labels"].astype(str):
        key = f"hs_point_{label}"
        if key in data:
            points[label] = np.asarray(data[key], dtype=float)
    return points


def _kpath_axis(points, path_labels, primitive_vectors):
    nodes_frac = [points[label] for label in path_labels]
    nodes_cart = _kfrac_to_cart(np.asarray(nodes_frac), primitive_vectors)
    x_nodes = [0.0]
    for i in range(1, len(nodes_cart)):
        x_nodes.append(x_nodes[-1] + float(np.linalg.norm(nodes_cart[i] - nodes_cart[i - 1])))
    return np.asarray(x_nodes), nodes_cart


def _project_kpoints_to_path(k_frac_points, points, path_labels, primitive_vectors, tol_cart=1.0e-6):
    k_frac_points = np.asarray(k_frac_points, dtype=float)
    dim = k_frac_points.shape[1]
    x_nodes, nodes_cart = _kpath_axis(points, path_labels, primitive_vectors)
    integer_shifts = [np.asarray(s, dtype=float) - 1.0 for s in np.ndindex(*([3] * dim))]
    projected = []
    for ik, q in enumerate(k_frac_points):
        best = None
        for shift in integer_shifts:
            q_equiv = q + shift
            q_cart = _kfrac_to_cart(np.asarray([q_equiv]), primitive_vectors)[0]
            for iseg in range(len(path_labels) - 1):
                a_cart = nodes_cart[iseg]
                b_cart = nodes_cart[iseg + 1]
                v = b_cart - a_cart
                denom = float(np.dot(v, v))
                if denom == 0.0:
                    continue
                t = float(np.dot(q_cart - a_cart, v) / denom)
                if t < -1.0e-10 or t > 1.0 + 1.0e-10:
                    continue
                closest = a_cart + t * v
                dist = float(np.linalg.norm(q_cart - closest))
                if dist <= tol_cart:
                    x = x_nodes[iseg] + t * float(np.linalg.norm(v))
                    candidate = (dist, ik, x, iseg, t, q_equiv)
                    if best is None or candidate[0] < best[0]:
                        best = candidate
        if best is not None:
            projected.append(best)
    if not projected:
        dim = k_frac_points.shape[1]
        return np.array([], dtype=int), np.array([]), np.empty((0, dim)), x_nodes
    return (
        np.asarray([p[1] for p in projected], dtype=int),
        np.asarray([p[2] for p in projected], dtype=float),
        np.asarray([p[5] for p in projected], dtype=float),
        x_nodes,
    )


path_labels = data["path_labels"].astype(str)
hs_points = _display_high_symmetry_points()
missing_points = [label for label in path_labels if label not in hs_points]
if missing_points:
    path_indices = data["path_k_indices"].astype(int)
    path_x = data["path_x"]
    path_q_equiv = data["path_q_equiv"]
    x_ticks = data["x_ticks"]
else:
    path_indices, path_x, path_q_equiv, x_ticks = _project_kpoints_to_path(
        data["k_frac_folded"], hs_points, path_labels, data["primitive_vectors"]
    )
x_labels = data["x_tick_labels"].astype(str)

all_energies = np.concatenate([
    data[f"evals_ev_spin_{spin}"] - float(data["ref_energy_ev"])
    for spin in spin_indices
])
finite_energies = all_energies[np.isfinite(all_energies)]
default_emin = float(np.floor(finite_energies.min())) if len(finite_energies) else -10.0
default_emax = float(np.ceil(finite_energies.max())) if len(finite_energies) else 10.0

spin_widget = ipw.Dropdown(options=spin_indices, value=spin_indices[0], description="Spin:")
marker_scale = ipw.FloatSlider(value=220.0, min=10.0, max=800.0, step=10.0, description="Scale:")
emin_widget = ipw.FloatText(value=default_emin, description="E min [eV]:", layout={"width": "180px"})
emax_widget = ipw.FloatText(value=default_emax, description="E max [eV]:", layout={"width": "180px"})
show_kmap = ipw.Checkbox(value=True, description="Show k-map")
out = ipw.Output()

def _plot(_=None):
    with out:
        out.clear_output(wait=True)
        spin = spin_widget.value
        weights = data[f"weights_spin_{spin}"]
        energies = data[f"evals_ev_spin_{spin}"] - float(data["ref_energy_ev"])
        emin = float(emin_widget.value)
        emax = float(emax_widget.value)
        if emin > emax:
            emin, emax = emax, emin

        if show_kmap.value and int(data["dim"]) == 2:
            k_frac = _canonical_frac(data["k_frac_folded"])
            path_q = _canonical_frac(path_q_equiv)
            fig, ax = plt.subplots(figsize=(4.5, 4.0))
            ax.scatter(k_frac[:, 0], k_frac[:, 1], s=18, alpha=0.45, label="folded k-points")
            if len(path_q):
                ax.scatter(path_q[:, 0], path_q[:, 1], s=42, color="tab:red", label="on plotted path")
            nodes = []
            for label in path_labels:
                if label in hs_points:
                    point = hs_points[label]
                    nodes.append(point)
                    ax.text(point[0], point[1], label, ha="center", va="bottom")
            if nodes:
                nodes = np.asarray(nodes)
                ax.plot(nodes[:, 0], nodes[:, 1], color="black", linewidth=1.2)
            ax.set_xlabel("primitive reciprocal fractional k1")
            ax.set_ylabel("primitive reciprocal fractional k2")
            ax.set_xlim(-0.03, 1.03)
            ax.set_ylim(-0.03, 1.03)
            ax.set_aspect("equal", adjustable="box")
            ax.legend(loc="upper right", fontsize=8)
            fig.tight_layout()
            plt.show()

        fig, ax = plt.subplots(figsize=(7, 4))
        energy_mask = (energies >= emin) & (energies <= emax)
        for ik, x in zip(path_indices, path_x):
            sizes = marker_scale.value * np.maximum(weights[ik], 0.0)
            mask = (sizes > 1.0e-8) & energy_mask
            ax.scatter(np.full(np.count_nonzero(mask), x), energies[mask], s=sizes[mask], alpha=0.7, color="black")
        for xt in x_ticks:
            ax.axvline(xt, linewidth=0.8, alpha=0.4)
        ax.axhline(0.0, linestyle="--", linewidth=1)
        ax.set_xticks(x_ticks, x_labels)
        ax.set_xlim(x_ticks[0], x_ticks[-1])
        ax.set_ylim(emin, emax)
        ax.set_xlabel("primitive-cell k-path")
        ax.set_ylabel("Energy - reference [eV]")
        ax.set_title(f"Unfolded weights, spin {spin}")
        fig.tight_layout()
        plt.show()

for widget in (spin_widget, marker_scale, emin_widget, emax_widget, show_kmap):
    widget.observe(_plot, "value")
display(ipw.HBox([spin_widget, marker_scale, emin_widget, emax_widget, show_kmap]), out)
_plot()
